# Sentinel-AML Data Exploration

This notebook provides initial data exploration and analysis for the Sentinel-AML system.

## Objectives
1. Understand transaction data patterns
2. Identify potential money laundering indicators
3. Explore graph relationships
4. Analyze risk factors

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set up plotting
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
%matplotlib inline

In [ ]:
# Import Sentinel-AML modules
import sys
sys.path.append('../src')

from sentinel_aml.data.models import Account, Transaction, TransactionType
from sentinel_aml.core.utils import (
    calculate_velocity_score,
    detect_round_dollar_pattern,
    is_high_risk_jurisdiction
)

## Sample Data Generation

Generate synthetic transaction data for exploration.

In [ ]:
# Generate sample transaction data
np.random.seed(42)

# Create sample accounts
accounts = []
for i in range(100):
    account = {
        'account_id': f'ACC{i:06d}',
        'customer_name': f'Customer_{i}',
        'account_type': np.random.choice(['checking', 'savings', 'business']),
        'risk_score': np.random.beta(2, 5),  # Skewed towards lower risk
        'country_code': np.random.choice(['US', 'CA', 'GB', 'DE', 'FR', 'IR', 'KP'], 
                                       p=[0.4, 0.15, 0.15, 0.1, 0.1, 0.05, 0.05]),
        'is_pep': np.random.choice([True, False], p=[0.05, 0.95]),
        'balance': np.random.lognormal(10, 1)
    }
    accounts.append(account)

accounts_df = pd.DataFrame(accounts)
print(f"Generated {len(accounts_df)} accounts")
accounts_df.head()

In [ ]:
# Generate sample transactions
transactions = []
base_time = datetime.now()

for i in range(1000):
    # Random account selection
    from_account = np.random.choice(accounts_df['account_id'])
    to_account = np.random.choice(accounts_df['account_id'])
    
    # Skip self-transactions
    if from_account == to_account:
        continue
    
    # Generate transaction amount (with some suspicious patterns)
    if np.random.random() < 0.1:  # 10% suspicious transactions
        # Potential structuring - amounts just under $10k
        amount = np.random.uniform(9000, 9999)
    else:
        # Normal transactions
        amount = np.random.lognormal(6, 1.5)
    
    transaction = {
        'transaction_id': f'TXN{i:06d}',
        'from_account_id': from_account,
        'to_account_id': to_account,
        'amount': round(amount, 2),
        'timestamp': base_time - timedelta(days=np.random.randint(0, 30)),
        'transaction_type': np.random.choice(['transfer', 'wire', 'ach', 'deposit']),
        'currency': 'USD',
        'is_cash': np.random.choice([True, False], p=[0.2, 0.8]),
        'is_international': np.random.choice([True, False], p=[0.3, 0.7])
    }
    transactions.append(transaction)

transactions_df = pd.DataFrame(transactions)
print(f"Generated {len(transactions_df)} transactions")
transactions_df.head()

## Exploratory Data Analysis

In [ ]:
# Basic statistics
print("Transaction Amount Statistics:")
print(transactions_df['amount'].describe())

print("\nAccount Risk Score Statistics:")
print(accounts_df['risk_score'].describe())

In [ ]:
# Visualize transaction amounts
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Transaction amount distribution
axes[0, 0].hist(transactions_df['amount'], bins=50, alpha=0.7)
axes[0, 0].set_title('Transaction Amount Distribution')
axes[0, 0].set_xlabel('Amount ($)')
axes[0, 0].set_ylabel('Frequency')

# Log scale for better visualization
axes[0, 1].hist(np.log10(transactions_df['amount']), bins=50, alpha=0.7)
axes[0, 1].set_title('Transaction Amount Distribution (Log Scale)')
axes[0, 1].set_xlabel('Log10(Amount)')
axes[0, 1].set_ylabel('Frequency')

# Risk score distribution
axes[1, 0].hist(accounts_df['risk_score'], bins=30, alpha=0.7)
axes[1, 0].set_title('Account Risk Score Distribution')
axes[1, 0].set_xlabel('Risk Score')
axes[1, 0].set_ylabel('Frequency')

# Transaction types
transaction_counts = transactions_df['transaction_type'].value_counts()
axes[1, 1].pie(transaction_counts.values, labels=transaction_counts.index, autopct='%1.1f%%')
axes[1, 1].set_title('Transaction Types')

plt.tight_layout()
plt.show()

## Suspicious Pattern Detection

In [ ]:
# Identify potential structuring (amounts just under $10k)
structuring_threshold = 10000
potential_structuring = transactions_df[
    (transactions_df['amount'] >= 9000) & 
    (transactions_df['amount'] < structuring_threshold)
]

print(f"Potential structuring transactions: {len(potential_structuring)}")
print(f"Percentage of total: {len(potential_structuring)/len(transactions_df)*100:.2f}%")

if len(potential_structuring) > 0:
    print("\nSample structuring transactions:")
    print(potential_structuring[['transaction_id', 'amount', 'from_account_id', 'to_account_id']].head())

In [ ]:
# Analyze transaction velocity by account
account_transaction_counts = transactions_df.groupby('from_account_id').size().reset_index(name='transaction_count')
high_velocity_accounts = account_transaction_counts[account_transaction_counts['transaction_count'] > 20]

print(f"High velocity accounts (>20 transactions): {len(high_velocity_accounts)}")

# Visualize transaction velocity
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.hist(account_transaction_counts['transaction_count'], bins=20, alpha=0.7)
plt.title('Transaction Count per Account')
plt.xlabel('Number of Transactions')
plt.ylabel('Number of Accounts')

plt.subplot(1, 2, 2)
plt.boxplot(account_transaction_counts['transaction_count'])
plt.title('Transaction Count Distribution')
plt.ylabel('Number of Transactions')

plt.tight_layout()
plt.show()

## Graph Analysis

In [ ]:
# Create transaction network graph
G = nx.DiGraph()

# Add nodes (accounts)
for _, account in accounts_df.iterrows():
    G.add_node(account['account_id'], 
               risk_score=account['risk_score'],
               account_type=account['account_type'],
               country_code=account['country_code'])

# Add edges (transactions)
for _, txn in transactions_df.iterrows():
    G.add_edge(txn['from_account_id'], txn['to_account_id'],
               amount=txn['amount'],
               transaction_id=txn['transaction_id'],
               timestamp=txn['timestamp'])

print(f"Graph created with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges")

In [ ]:
# Analyze graph properties
print("Graph Analysis:")
print(f"Number of nodes: {G.number_of_nodes()}")
print(f"Number of edges: {G.number_of_edges()}")
print(f"Graph density: {nx.density(G):.4f}")
print(f"Number of connected components: {nx.number_weakly_connected_components(G)}")

# Calculate centrality measures
in_degree_centrality = nx.in_degree_centrality(G)
out_degree_centrality = nx.out_degree_centrality(G)
betweenness_centrality = nx.betweenness_centrality(G)

# Find most central accounts
top_in_degree = sorted(in_degree_centrality.items(), key=lambda x: x[1], reverse=True)[:5]
top_out_degree = sorted(out_degree_centrality.items(), key=lambda x: x[1], reverse=True)[:5]
top_betweenness = sorted(betweenness_centrality.items(), key=lambda x: x[1], reverse=True)[:5]

print("\nTop 5 accounts by in-degree centrality (money receivers):")
for account, centrality in top_in_degree:
    print(f"  {account}: {centrality:.4f}")

print("\nTop 5 accounts by out-degree centrality (money senders):")
for account, centrality in top_out_degree:
    print(f"  {account}: {centrality:.4f}")

print("\nTop 5 accounts by betweenness centrality (intermediaries):")
for account, centrality in top_betweenness:
    print(f"  {account}: {centrality:.4f}")

In [ ]:
# Visualize a subset of the graph
# Select top 20 most connected accounts for visualization
degree_dict = dict(G.degree())
top_accounts = sorted(degree_dict.items(), key=lambda x: x[1], reverse=True)[:20]
top_account_ids = [account[0] for account in top_accounts]

# Create subgraph
subgraph = G.subgraph(top_account_ids)

# Visualize
plt.figure(figsize=(15, 10))
pos = nx.spring_layout(subgraph, k=1, iterations=50)

# Color nodes by risk score
node_colors = [accounts_df[accounts_df['account_id'] == node]['risk_score'].iloc[0] 
               if len(accounts_df[accounts_df['account_id'] == node]) > 0 else 0
               for node in subgraph.nodes()]

# Draw the graph
nx.draw(subgraph, pos, 
        node_color=node_colors, 
        node_size=300,
        cmap=plt.cm.Reds,
        with_labels=True,
        font_size=8,
        arrows=True,
        edge_color='gray',
        alpha=0.7)

plt.title('Transaction Network (Top 20 Most Connected Accounts)\nNode color represents risk score')
plt.colorbar(plt.cm.ScalarMappable(cmap=plt.cm.Reds), label='Risk Score')
plt.show()

## Risk Factor Analysis

In [ ]:
# Analyze risk factors
risk_analysis = accounts_df.copy()

# Add transaction-based features
account_stats = transactions_df.groupby('from_account_id').agg({
    'amount': ['count', 'sum', 'mean', 'std'],
    'is_cash': 'sum',
    'is_international': 'sum'
}).round(2)

account_stats.columns = ['txn_count', 'total_amount', 'avg_amount', 'amount_std', 'cash_txns', 'intl_txns']
account_stats = account_stats.reset_index()

# Merge with account data
risk_analysis = risk_analysis.merge(account_stats, left_on='account_id', right_on='from_account_id', how='left')
risk_analysis = risk_analysis.fillna(0)

# Add high-risk jurisdiction flag
risk_analysis['high_risk_jurisdiction'] = risk_analysis['country_code'].apply(is_high_risk_jurisdiction)

print("Risk Analysis Summary:")
print(f"Accounts in high-risk jurisdictions: {risk_analysis['high_risk_jurisdiction'].sum()}")
print(f"PEP accounts: {risk_analysis['is_pep'].sum()}")
print(f"Average risk score: {risk_analysis['risk_score'].mean():.3f}")

risk_analysis.head()

In [ ]:
# Correlation analysis
correlation_features = ['risk_score', 'txn_count', 'total_amount', 'avg_amount', 
                       'cash_txns', 'intl_txns', 'is_pep', 'high_risk_jurisdiction']

correlation_matrix = risk_analysis[correlation_features].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, fmt='.3f')
plt.title('Risk Factor Correlation Matrix')
plt.tight_layout()
plt.show()

## Key Insights and Recommendations

Based on this exploratory analysis, we can identify several patterns and recommendations for the Sentinel-AML system:

### Findings:
1. **Transaction Patterns**: The majority of transactions follow normal patterns, but we identified potential structuring activities
2. **Network Structure**: The transaction network shows varying levels of connectivity, with some accounts acting as hubs
3. **Risk Factors**: Multiple risk factors can be combined to create comprehensive risk scores

### Recommendations:
1. **Implement velocity-based detection** for accounts with unusually high transaction frequencies
2. **Use graph centrality measures** to identify potential money laundering intermediaries
3. **Combine multiple risk factors** for more accurate risk assessment
4. **Focus on structuring detection** for transactions near reporting thresholds

### Next Steps:
1. Develop GNN models for graph-based fraud detection
2. Implement real-time pattern detection algorithms
3. Create automated alert generation based on risk thresholds
4. Build explainable AI components for regulatory compliance